# 07 – Shopify Export

Transform validated, platform-independent product data into a Shopify-compatible
import dataset.

## Responsibility

This notebook:

- loads validated product data from the transformation pipeline
- loads normalized image metadata from the image-processing pipeline
- transforms product and variant data into Shopify's import structure
- prepares product descriptions, prices, SKUs and image references
- validates the final Shopify export before saving it
- performs no Shopify API mutations

## Input

- `valid_products.csv`
- normalized image metadata from the image-processing pipeline

## Output

- Shopify-compatible product import data

Shopify API uploads and synchronization are intentionally kept separate from
this export pipeline.

## Setup

Define the supplier, input files and Shopify export location in one place so
the export logic remains easy to reuse and maintain.

In [ ]:
from pathlib import Path
import pandas as pd

SUPPLIER = "snickers"

DATA_DIR = Path("../data") / SUPPLIER
PRODUCT_DATA_PATH = DATA_DIR / "valid_products.csv"
IMAGE_METADATA_PATH = DATA_DIR / "normalized_image_metadata.csv"

SHOPIFY_DIR = Path("../data/shopify")
SHOPIFY_DIR.mkdir(parents=True, exist_ok=True)

SHOPIFY_EXPORT_PATH = SHOPIFY_DIR / "shopify_products.csv"

## Load Export Data

Load the validated product data and normalized image metadata produced by the
previous pipeline steps.

The notebook stops early if either required input file is missing.

In [ ]:
for input_path in (PRODUCT_DATA_PATH, IMAGE_METADATA_PATH):
    if not input_path.exists():
        raise FileNotFoundError(
            f"Required input file not found: {input_path}"
        )

products = pd.read_csv(
    PRODUCT_DATA_PATH,
    dtype={
        "product_id": "string",
        "variant_sku": "string",
        "ean": "string",
    },
)

product_images = pd.read_csv(
    IMAGE_METADATA_PATH,
    dtype={
        "product_id": "string",
        "variant_sku": "string",
    },
)

print(f"Loaded products: {len(products):,}")
print(f"Loaded image relationships: {len(product_images):,}")

## Validate Export Inputs

Validate the product and image datasets before applying Shopify-specific
transformations.

In [ ]:
assert products["product_id"].notna().all(), (
    "Missing product IDs found."
)

assert products["variant_sku"].notna().all(), (
    "Missing variant SKUs found."
)

assert not products["variant_sku"].duplicated().any(), (
    "Duplicate variant SKUs found."
)

assert product_images["normalized_image_path"].notna().all(), (
    "Missing normalized image paths found."
)

print("Export input validation passed.")

## Build Shopify Product Handles

Shopify groups product variants using a shared product handle.

Create a stable handle from the normalized product name and product ID so every
variant belonging to the same product receives the same handle.

In [ ]:
import re


def create_shopify_handle(product_name, product_id):
    handle = str(product_name).strip().lower()

    handle = re.sub(r"[^a-z0-9åäö]+", "-", handle)
    handle = handle.strip("-")

    return f"{handle}-{product_id}"


products["handle"] = products.apply(
    lambda row: create_shopify_handle(
        row["product_name"],
        row["product_id"],
    ),
    axis=1,
)

assert products.groupby("product_id")["handle"].nunique().max() == 1

print(f"Unique Shopify products: {products['handle'].nunique():,}")

## Build Shopify Variant Data

Map the normalized product fields to the Shopify variant structure.

Each normalized row represents one sellable variant while variants belonging
to the same product share the same Shopify handle.

In [ ]:
shopify_variants = pd.DataFrame({
    "Handle": products["handle"],
    "Title": products["product_name"],
    "Option1 Name": "Color",
    "Option1 Value": products["color"],
    "Option2 Name": "Size",
    "Option2 Value": products["size"],
    "Variant SKU": products["variant_sku"],
    "Variant Price": products["price_sek"],
    "Variant Barcode": products["ean"],
})

print(f"Shopify variants prepared: {len(shopify_variants):,}")

shopify_variants.head()

## Build Shopify Product Content

Prepare the product-level content used by Shopify.

Product descriptions are shared by all variants belonging to the same product.

In [ ]:
shopify_variants["Body (HTML)"] = products["description"]

assert (
    shopify_variants.groupby("Handle")["Body (HTML)"].nunique().max()
    <= 1
), "Multiple descriptions found for the same Shopify product."

print("Shopify product content prepared.")

## Prepare Shopify Image Mapping

Prepare the relationship between Shopify products, variants and their normalized
image files.

The local normalized image paths are preserved as export metadata. Uploading
images to Shopify and replacing these paths with Shopify-hosted URLs belongs to
the separate synchronization step.

In [ ]:
image_export = product_images[
    [
        "product_id",
        "variant_sku",
        "image_url",
        "normalized_image_path",
    ]
].copy()

image_export = image_export.merge(
    products[
        [
            "product_id",
            "variant_sku",
            "handle",
        ]
    ],
    how="left",
    on=["product_id", "variant_sku"],
    validate="many_to_one",
)

assert image_export["handle"].notna().all(), (
    "Some image relationships could not be matched to a Shopify product."
)

print(f"Shopify image relationships prepared: {len(image_export):,}")

## Build Shopify Export

Combine the prepared product and variant fields into the final Shopify export
dataset.

Each row represents one product variant while product-level fields remain
consistent across variants sharing the same handle.

In [ ]:
shopify_export = shopify_variants[
    [
        "Handle",
        "Title",
        "Body (HTML)",
        "Option1 Name",
        "Option1 Value",
        "Option2 Name",
        "Option2 Value",
        "Variant SKU",
        "Variant Price",
        "Variant Barcode",
    ]
].copy()

assert len(shopify_export) == len(products), (
    "Shopify export row count does not match the validated product data."
)

print(f"Shopify export rows prepared: {len(shopify_export):,}")

## Validate Shopify Export

Validate the final Shopify dataset before writing it to disk.

In [ ]:
assert shopify_export["Handle"].notna().all(), (
    "Missing Shopify handles found."
)

assert shopify_export["Variant SKU"].notna().all(), (
    "Missing variant SKUs found."
)

assert not shopify_export["Variant SKU"].duplicated().any(), (
    "Duplicate variant SKUs found."
)

assert shopify_export["Variant Price"].notna().all(), (
    "Missing variant prices found."
)

assert shopify_export["Option1 Value"].notna().all(), (
    "Missing variant colors found."
)

assert shopify_export["Option2 Value"].notna().all(), (
    "Missing variant sizes found."
)

print(f"Validated Shopify products: {shopify_export['Handle'].nunique():,}")
print(f"Validated Shopify variants: {len(shopify_export):,}")
print("Shopify export validation passed.")

## Export Shopify Data

Write the validated Shopify product and variant dataset to disk.

This file contains no API operations and can be inspected independently before
any Shopify synchronization is performed.

In [ ]:
shopify_export.to_csv(
    SHOPIFY_EXPORT_PATH,
    index=False,
)

assert SHOPIFY_EXPORT_PATH.exists(), (
    "Shopify export file was not created."
)

print(
    f"Exported {len(shopify_export):,} Shopify variants "
    f"to {SHOPIFY_EXPORT_PATH}"
)

## Final Validation

Perform final integrity checks on the exported Shopify dataset before it is
passed to a separate synchronization process.

In [ ]:
exported_shopify_data = pd.read_csv(
    SHOPIFY_EXPORT_PATH,
    dtype={
        "Variant SKU": "string",
        "Variant Barcode": "string",
    },
)

assert len(exported_shopify_data) == len(shopify_export), (
    "Saved Shopify export row count does not match the prepared dataset."
)

assert exported_shopify_data["Variant SKU"].notna().all(), (
    "Missing variant SKUs found in the saved export."
)

assert not exported_shopify_data["Variant SKU"].duplicated().any(), (
    "Duplicate variant SKUs found in the saved export."
)

print(f"Final Shopify products: {exported_shopify_data['Handle'].nunique():,}")
print(f"Final Shopify variants: {len(exported_shopify_data):,}")
print("Final Shopify export validation passed.")